# 03 -- Promotion audit log

Read `logs/decisions.jsonl` and render the full retrain history. Each
row is one promotion decision (or a rollback / bootstrap event).

Run `02_simulate_production_8wk.ipynb` first to populate the log.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from modelgate import audit

entries = audit.read_all()
print(f'{len(entries)} entries')

In [ ]:
rows = []
for e in entries:
    row = {
        'recorded_at': e.get('recorded_at', '')[:19],
        'action': e['action'],
        'week': e.get('week'),
        'version': e.get('challenger_version') or e.get('version') or e.get('to_version'),
    }
    if 'challenger_metrics' in e:
        row['chal_auc'] = round(e['challenger_metrics']['roc_auc'], 4)
        row['champ_auc'] = round(e['champion_metrics']['roc_auc'], 4)
    if 'decision' in e:
        failed = [c['name'] for c in e['decision']['checks'] if not c['passed']]
        row['failed_checks'] = ','.join(failed) if failed else '-'
    rows.append(row)
df = pd.DataFrame(rows)
df

## Promote vs skip ratio

In [ ]:
df['action'].value_counts()

## Which gate fires most often?

In [ ]:
from collections import Counter
fail_counter = Counter()
for e in entries:
    if 'decision' in e:
        for c in e['decision']['checks']:
            if not c['passed']:
                fail_counter[c['name']] += 1
pd.Series(fail_counter).sort_values(ascending=False)

## Drilling into one decision

Pick the first skip and see every gate value.

In [ ]:
import json
first_skip = next((e for e in entries if e['action'] == 'skip'), None)
if first_skip:
    print(json.dumps(first_skip['decision'], indent=2, default=str))